#### Libraries, utils, plot settings

In [77]:
# =========================
# config.py
# =========================
from dataclasses import dataclass
from typing import Optional

@dataclass
class SimulationConfig:
    dt: float = 0.1
    T: float = 5000.0
    T_transient: float = 1000.0
    noise_std: float = 0.0
    device: str = "cpu"
    seed: Optional[int] = 0

@dataclass
class AnalysisConfig:
    max_lag: int = 2000
    fixed_point_var_thresh: float = 1e-4
    fixed_point_deriv_thresh: float = 1e-5


# =========================
# networks.py
# =========================
import torch
import numpy as np
from scipy.stats import levy_stable

def sompolinsky_network(N: int, g: float, device="cpu") -> torch.Tensor:
    J = torch.randn(N, N, device=device) * (g / np.sqrt(N))
    J.fill_diagonal_(0.0)
    return J

def uniform_self_coupling_network(N: int, g: float, s: float, device="cpu") -> torch.Tensor:
    J = sompolinsky_network(N, g, device)
    J += torch.eye(N, device=device) * s
    return J

def distributed_self_coupling_network(N: int, g: float, mu: float, sigma: float, device="cpu") -> torch.Tensor:
    J = sompolinsky_network(N, g, device)
    self_couplings = torch.from_numpy(np.random.lognormal(mean=mu, sigma=sigma, size=N)).to(device)
    J += torch.diag(self_couplings)
    return J

def alpha_stable_network(N: int, g: float, alpha: float, device="cpu") -> torch.Tensor:
    scale = g / (N ** (1 / alpha))
    samples = levy_stable.rvs(alpha=alpha, beta=0, scale=scale, size=(N, N))
    J = torch.tensor(samples, dtype=torch.float32, device=device)
    J.fill_diagonal_(0.0)
    return J

def structural_strength_network(N: int, g: float, lognorm_mu: float, lognorm_sigma: float, gamma: float, device="cpu") -> torch.Tensor:
    degrees = np.clip(np.random.lognormal(lognorm_mu, lognorm_sigma, size=N).astype(int), 1, N-1)
    A = np.zeros((N, N))
    stubs = [i for i, k in enumerate(degrees) for _ in range(k)]
    np.random.shuffle(stubs)
    for i in range(0, len(stubs)-1, 2):
        u, v = stubs[i], stubs[i+1]
        if u != v:
            A[u, v] += 1
            A[v, u] += 1
    A = (A > 0).astype(float)
    W = np.random.randn(N, N) * (g / np.sqrt(degrees.mean()))
    W_sym = gamma * W + np.sqrt(1 - gamma**2) * W.T
    J = A * W_sym
    np.fill_diagonal(J, 0.0)
    return torch.tensor(J, dtype=torch.float32, device=device)


# =========================
# dynamics.py
# =========================
import torch
from typing import Tuple
from config import SimulationConfig

def simulate_rnn(J: torch.Tensor, sim_cfg: SimulationConfig) -> Tuple[torch.Tensor, torch.Tensor]:
    if sim_cfg.seed is not None:
        torch.manual_seed(sim_cfg.seed)
    dt = sim_cfg.dt
    T_steps = int(sim_cfg.T / dt)
    N = J.shape[0]
    x = torch.zeros(T_steps, N, device=sim_cfg.device)
    dx = torch.zeros_like(x)
    x[0] = 0.1 * torch.randn(N, device=sim_cfg.device)

    for t in range(T_steps-1):
        phi = torch.tanh(x[t])
        deterministic = -x[t] + J @ phi
        dx[t] = deterministic
        noise = sim_cfg.noise_std * torch.randn(N, device=sim_cfg.device)
        x[t+1] = x[t] + dt * deterministic + torch.sqrt(torch.tensor(dt)) * noise

    return x, dx


# =========================
# checks.py
# =========================
import torch
from config import AnalysisConfig

def is_fixed_point(x: torch.Tensor, dx: torch.Tensor, cfg: AnalysisConfig) -> bool:
    var = torch.var(x, dim=0).mean()
    deriv_mse = torch.mean(dx**2)
    return var < cfg.fixed_point_var_thresh or deriv_mse < cfg.fixed_point_deriv_thresh


# =========================
# analysis.py
# =========================
import torch
import numpy as np
from statsmodels.tsa.stattools import acf

def acf_stattools(X: torch.Tensor, max_lag: int) -> torch.Tensor:
    T, N = X.shape
    X_np = X.detach().cpu().numpy()
    acf_all = np.empty((max_lag, N))
    for i in range(N):
        xi = X_np[:, i] - X_np[:, i].mean()
        acf_all[:, i] = acf(xi, nlags=max_lag-1, fft=True, adjusted=True)
    return torch.tensor(acf_all, dtype=X.dtype, device=X.device)

def timescale_hwhm(acf: torch.Tensor) -> torch.Tensor:
    N = acf.shape[1]
    tau = torch.full((N,), torch.nan, device=acf.device)
    for i in range(N):
        below = torch.where(acf[:, i] < 0.5)[0]
        if len(below) > 0:
            tau[i] = float(below[0])
    return tau

def timescale_integrated(acf: torch.Tensor, dt: float) -> torch.Tensor:
    N = acf.shape[1]
    tau = torch.full((N,), torch.nan, device=acf.device)
    for i in range(N):
        zero_cross = torch.where(acf[:, i] <= 0)[0]
        if len(zero_cross) > 0:
            t0 = zero_cross[0]
            tau[i] = torch.sum(acf[:t0, i]) * dt
    return tau


# =========================
# io_utils.py
# =========================
import json
import torch
import numpy as np
from pathlib import Path
from typing import Dict

def save_results(outdir: Path, metadata: Dict, timescales_hwhm: torch.Tensor, timescales_int: torch.Tensor):
    outdir.mkdir(parents=True, exist_ok=True)
    np.savez(outdir / "timescales.npz", hwhm=timescales_hwhm.cpu().numpy(), integrated=timescales_int.cpu().numpy())
    with open(outdir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)


# =========================
# experiment.py
# =========================
from pathlib import Path
import torch

def run_experiment(J: torch.Tensor, sim_cfg: SimulationConfig, ana_cfg: AnalysisConfig, outdir: Path, metadata: dict):
    x, dx = simulate_rnn(J, sim_cfg)
    T_trans = int(sim_cfg.T_transient / sim_cfg.dt)
    x_stat, dx_stat = x[T_trans:], dx[T_trans:]
    metadata["fixed_point"] = is_fixed_point(x_stat, dx_stat, ana_cfg)

    if metadata["fixed_point"]:
        tau_hwhm = tau_int = torch.full((J.shape[0],), torch.nan)
    else:
        acf_vals = acf_stattools(x_stat, ana_cfg.max_lag)
        tau_hwhm = timescale_hwhm(acf_vals) * sim_cfg.dt
        tau_int = timescale_integrated(acf_vals, sim_cfg.dt)

    save_results(outdir, metadata, tau_hwhm, tau_int)

#### Vanilla network analysis

In [ ]:
# example_run.py
import torch
from pathlib import Path
from config import SimulationConfig, AnalysisConfig
from networks import sompolinsky_network
from experiment import run_experiment

# -------------------------------------------------
# Device selection
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------------------------------
# Global configuration
# -------------------------------------------------
sim_cfg = SimulationConfig(
    T=2000.0,
    T_transient=300.0,
    dt=0.1,
    device=device
)

ana_cfg = AnalysisConfig(
    max_lag=10000
)

# -------------------------------------------------
# Experiment parameters
# -------------------------------------------------
N_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
g = 1.5
n_repeats = 100
base_seed = 821991
base_outdir = Path("/data/zenari/RNN_timescales_comparison/results/sompolinsky")

# -------------------------------------------------
# Run experiments
# -------------------------------------------------
for N in N_list:
    print(f"\nRunning N = {N}")

    for run_idx in range(n_repeats):
        seed = base_seed + run_idx
        print(f"  Run {run_idx} (seed={seed})")

        # Update seed in config
        sim_cfg.seed = seed
        torch.manual_seed(seed)

        # Build network
        J = sompolinsky_network(N=N, g=g, device=device)

        # Metadata for output
        metadata = {
            "model": "Sompolinsky",
            "N": N,
            "g": g,
            "dt": sim_cfg.dt,
            "T": sim_cfg.T,
            "T_transient": sim_cfg.T_transient,
            "run_idx": run_idx,
            "seed": seed
        }

        # Output directory per run
        outdir = base_outdir / f"N_{N}" / f"run_{run_idx}"
        run_experiment(J, sim_cfg, ana_cfg, outdir, metadata)

print("\nAll experiments completed successfully.")

#### Alpha-stable network

In [103]:
# example_run_alpha_stable.py
import torch
from pathlib import Path

from config import SimulationConfig, AnalysisConfig
from networks import alpha_stable_network
from experiment import run_experiment

# -------------------------------------------------
# Device
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------------------------------
# Global configs
# -------------------------------------------------
sim_cfg = SimulationConfig(
    T=2000,
    T_transient=300,
    dt=0.1,
    device=device
)

ana_cfg = AnalysisConfig(
    max_lag=10000
)

# -------------------------------------------------
# Experiment parameters
# -------------------------------------------------
N_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10_000]
g = 1.8
alpha = 1.2
n_repeats = 10              # number of independent runs
base_seed = 821991          # reproducibility

base_outdir = Path("/data/zenari/RNN_timescales_comparison/results/alpha_stable")

# -------------------------------------------------
# Run experiment
# -------------------------------------------------
for N in N_list:
    print(f"\nRunning N = {N}")

    for run_idx in range(10, 10 + n_repeats):
        seed = base_seed + run_idx
        print(f"  Run {run_idx} (seed={seed})")

        # Set seed explicitly for reproducibility
        sim_cfg.seed = seed
        torch.manual_seed(seed)

        # Build alpha-stable network (new realization each run)
        J = alpha_stable_network(N=N, g=g, alpha=alpha, device=device)

        # Metadata
        metadata = {
            "model": "Alpha-stable",
            "N": N,
            "g": g,
            "alpha": alpha,
            "dt": sim_cfg.dt,
            "T": sim_cfg.T,
            "T_transient": sim_cfg.T_transient,
            "run_idx": run_idx,
            "seed": seed
        }

        # Output directory
        outdir = base_outdir / f"N_{N}" / f"run_{run_idx}"
        run_experiment(J, sim_cfg, ana_cfg, outdir, metadata)

print("\nAlpha-stable scaling + repetition experiment complete.")

#### Heterogeneous self-coupling experiment

In [142]:
# example_run.py
import torch
from pathlib import Path

# -------------------------------------------------
# Device
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------------------------------
# Global configs
# -------------------------------------------------
sim_cfg = SimulationConfig(
    T=2000,
    T_transient=300,
    dt=0.1,
    device=device
)

ana_cfg = AnalysisConfig(
    max_lag=10000
)

# -------------------------------------------------
# Experiment parameters
# -------------------------------------------------
N_list = [1000,2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10_000]
mu = 0.6
sigma = 0.6
g = 3
n_repeats = 20             # <-- number of independent runs
base_seed = 821991         # <-- reproducibility

base_outdir = Path("/data/zenari/RNN_timescales_comparison/results/self_couplings")
# -------------------------------------------------
# Run experiment
# -------------------------------------------------
for N in N_list:
    print(f"\nRunning N = {N}")

    for run_idx in range(30, n_repeats+30):
        seed = base_seed + run_idx

        print(f"  Run {run_idx} (seed={seed})")

        # Set seed explicitly for this run
        sim_cfg.seed = seed
        torch.manual_seed(seed)

        # Build network (new realization each run)
        J =  distributed_self_coupling_network(
            N=N, g=g, mu=mu, sigma=sigma, device=device
        )

        metadata = {
            "model": "Self-coupling distributed",
            "N": N,
            "g": g,
            "mu": mu,
            "sigma": sigma,
            "dt": sim_cfg.dt,
            "T": sim_cfg.T,
            "T_transient": sim_cfg.T_transient,
            "run_idx": run_idx,
            "seed": seed
        }

        outdir = base_outdir / f"N_{N}" / f"run_{run_idx}"
        run_experiment(J, sim_cfg, ana_cfg, outdir, metadata)

print("\nScaling + repetition experiment complete.")

#### Structural heterogeneity

In [ ]:
# example_run_structural_strength.py
import torch
from pathlib import Path

from config import SimulationConfig, AnalysisConfig
from networks import structural_strength_network
from experiment import run_experiment

# -------------------------------------------------
# Device
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -------------------------------------------------
# Global configs
# -------------------------------------------------
sim_cfg = SimulationConfig(
    T=2000,
    T_transient=300,
    dt=0.1,
    device=device
)

ana_cfg = AnalysisConfig(
    max_lag=10000
)

# -------------------------------------------------
# Experiment parameters
# -------------------------------------------------
N_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10_000]
mu = 3.0
sigma = 1.0
gamma = 0.4
g = 3.0
n_repeats = 20              # number of independent runs
base_seed = 821991          # reproducibility

base_outdir = Path("/PLACEHOLDER/results/structural_strength")

# -------------------------------------------------
# Run experiment
# -------------------------------------------------
for N in N_list:
    print(f"\nRunning N = {N}")

    for run_idx in range(10, 10 + n_repeats):
        seed = base_seed + run_idx
        print(f"  Run {run_idx} (seed={seed})")

        # Set seed explicitly for reproducibility
        sim_cfg.seed = seed
        torch.manual_seed(seed)

        # Build structural strength network
        J = structural_strength_network(
            N=N,
            g=g,
            lognorm_mu=mu,
            lognorm_sigma=sigma,
            gamma=gamma,
            device=device
        )

        # Metadata
        metadata = {
            "model": "Structural strength",
            "N": N,
            "g": g,
            "lognorm_mu": mu,
            "lognorm_sigma": sigma,
            "gamma": gamma,
            "dt": sim_cfg.dt,
            "T": sim_cfg.T,
            "T_transient": sim_cfg.T_transient,
            "run_idx": run_idx,
            "seed": seed
        }

        # Output directory
        outdir = base_outdir / f"N_{N}" / f"run_{run_idx}"
        run_experiment(J, sim_cfg, ana_cfg, outdir, metadata)

print("\nStructural strength scaling + repetition experiment complete.")

### Final plot

In [4]:
# plot_HWHM_cv.py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------------------------------
# Configuration for datasets
# -------------------------------------------------
datasets = {
    "sompolinsky": "/PLACEHOLDER/results/sompolinsky",
    "alpha_stable": "/PLACEHOLDER/results/alpha_stable",
    "self_couplings": "/PLACEHOLDER/results/self_couplings",
    "structural_strength": "/PLACEHOLDER/results/structural_strength",
}

dataset_labels = {
    "sompolinsky": "Fully",
    "alpha_stable": "Heavy tailed",
    "self_couplings": "Self-couplings",
    "structural_strength": "Heterogeneous",
}

n_repeats_dict = {
    "sompolinsky": 100,
    "alpha_stable": 20,
    "self_couplings": 50,
    "structural_strength": 50,
}

N_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

# -------------------------------------------------
# Collect HWHM CV data
# -------------------------------------------------
all_mean_cv = {}
all_sem_cv = {}

for name, base_dir_str in datasets.items():
    base_dir = Path(base_dir_str)
    n_repeats = n_repeats_dict[name]

    cv_runs_all = []

    for N in N_list:
        cv_runs = []
        for run_idx in range(n_repeats):
            data_path = base_dir / f"N_{N}" / f"run_{run_idx}" / "timescales.npz"
            if not data_path.exists():
                continue

            data = np.load(data_path)
            tau = data["hwhm"]
            tau = tau[np.isfinite(tau)]
            if len(tau) > 0:
                cv_runs.append(np.std(tau) / np.mean(tau))

        cv_runs_all.append(cv_runs)

    # Compute mean and SEM per network size
    mean_cv = [np.nan if len(runs) == 0 else np.mean(runs) for runs in cv_runs_all]
    sem_cv = [np.nan if len(runs) == 0 else np.std(runs, ddof=1) / np.sqrt(len(runs)) for runs in cv_runs_all]

    all_mean_cv[name] = mean_cv
    all_sem_cv[name] = sem_cv

# -------------------------------------------------
# Plotting function
# -------------------------------------------------
def plot_HWHM_cv(fig_size=(8, 6), save_path="HWHM_cv_overlay.png"):
    plt.figure(figsize=fig_size)
    for name in datasets:
        plt.errorbar(
            N_list,
            all_mean_cv[name],
            yerr=all_sem_cv[name],
            fmt="o-",
            lw=2,
            capsize=3,
            label=dataset_labels[name]
        )
    plt.xlabel("Network size N")
    plt.ylabel(r"CV($\tau$) [HWHM]")
    plt.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=len(datasets),
        frameon=False
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=1000, bbox_inches="tight")
    plt.close()  # Close figure to free memory

# -------------------------------------------------
# Generate figures
# -------------------------------------------------
# Small figure for main text
plot_HWHM_cv(
    fig_size=(8/2.54, 6/2.54),
    save_path="PLACEHOLDER/HWHM_cv_overlay_small.png"
)

# Large figure for supplement
plot_HWHM_cv(
    fig_size=(16/2.54, 12/2.54),
    save_path="PLACEHOLDER/HWHM_cv_overlay_large.png"
)

print("HWHM CV plots saved successfully.")

In [3]:
def plot_HWHM_cv(fig_size, save_path, legend_position="top"):
    plt.figure(figsize=fig_size)
    
    # Plot all datasets
    for name in datasets:
        plt.errorbar(
            N_list,
            all_mean_cv[name],
            yerr=all_sem_cv[name],
            fmt="o-",
            lw=2,
            capsize=3,
            label=dataset_labels[name]
        )
    
    plt.xlabel("Network size N")
    plt.ylabel(r"CV($\tau$) [HWHM]")

    # Legend placement
    if legend_position == "top":
        plt.legend(
            loc="lower center",
            bbox_to_anchor=(0.5, 1.15),  # raise legend above plot
            ncol=len(datasets),
            frameon=False
        )
        plt.subplots_adjust(top=0.85)  # leave space for legend
    elif legend_position == "right":
        plt.legend(
            loc="center left",
            bbox_to_anchor=(1, 0.5),
            frameon=False
        )
        plt.subplots_adjust(right=0.75)  # leave space on right for legend
    else:  # inside plot, default top-right
        plt.legend(loc="upper right", frameon=False)

    plt.tight_layout(pad=0.5)
    plt.savefig(save_path, dpi=1000, bbox_inches="tight")
    plt.close()

# Small figure for main text, legend on top
plot_HWHM_cv(
    fig_size=(8/2.54, 6/2.54),
    save_path="/home/zenari/RNN_heterogeneity/results/HWHM_cv_overlay_small.png",
    legend_position="top"
)

# Large figure for supplement, legend to the right
plot_HWHM_cv(
    fig_size=(16/2.54, 12/2.54),
    save_path="/home/zenari/RNN_heterogeneity/results/HWHM_cv_overlay_large.png",
    legend_position="right"
)
